<a href="https://colab.research.google.com/github/Sushanth4079/NLP/blob/main/4079_NLP_ASS_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Importing libraries**

In [2]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
import numpy as np


**LOADING ELMO MODEL**

In [3]:
elmo=hub.load("https://tfhub.dev/google/elmo/3")

**TEXT CORPUS**

In [6]:
sentences=['The bank will not approve the loan',
           'He sat on the river bank']

**Generate embeddings**

In [7]:
embeddings = elmo.signatures['default'](tf.constant(sentences))['elmo']
print(embeddings.shape)

(2, 7, 1024)


**Inspect Word Embedding (“bank”)**

In [8]:
# Convert sentences into tokens
tokenized = [sentence.split() for sentence in sentences]

# Find index of "bank"
idx1 = tokenized[0].index("bank")
idx2 = tokenized[1].index("bank")

# Extract embeddings
bank_emb_1 = embeddings[0][idx1]
bank_emb_2 = embeddings[1][idx2]

print("First 10 values (Sentence 1):", bank_emb_1[:10])
print("First 10 values (Sentence 2):", bank_emb_2[:10])

First 10 values (Sentence 1): tf.Tensor(
[-0.9086765  -0.36578754 -0.07339765  0.586675    0.22132948 -0.7657321
 -0.11816446 -0.30227882 -0.06137528  0.1695182 ], shape=(10,), dtype=float32)
First 10 values (Sentence 2): tf.Tensor(
[-0.15614763  0.28345656 -0.08914638  0.2069506   0.1524184  -0.2037305
 -0.10812807  0.03747292  0.5214868   0.35352594], shape=(10,), dtype=float32)


**Compare Context (Cosine Similarity)**

In [9]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(bank_emb_1.numpy(), bank_emb_2.numpy())

print("Similarity between 'bank' meanings:", sim)

Similarity between 'bank' meanings: 0.57813895


**TASK 2 WITH DIFFERENT TEXT CORPUS**

In [11]:
sentences2=['''the bat is flying
he hit the ball with a bat
''']

**Generate embeddings**

In [15]:
embeddings = elmo.signatures['default'](tf.constant(sentences2))['elmo']
print(embeddings.shape)

(1, 10, 1024)


**Inspect Word Embedding (“bat”)**

In [17]:
# Split the single multi-line string in sentences2 into a list of individual sentences
# sentences2 is ['''the bat is flying\nhe hit the ball with a bat\n''']
# The strip() removes leading/trailing whitespace including the final newline
parsed_sentences_for_bat = sentences2[0].strip().split('\n')
# parsed_sentences_for_bat will be ['the bat is flying', 'he hit the ball with a bat']

# Generate new embeddings using these 'bat' sentences
embeddings_for_bat = elmo.signatures['default'](tf.constant(parsed_sentences_for_bat))['elmo']

# Convert the parsed sentences into tokens
tokenized_bat_sents = [sentence.split() for sentence in parsed_sentences_for_bat]

# Find index of "bat" in each sentence
idx1 = tokenized_bat_sents[0].index("bat")
idx2 = tokenized_bat_sents[1].index("bat")

# Extract embeddings for "bat"
bat_emb_1 = embeddings_for_bat[0][idx1]
bat_emb_2 = embeddings_for_bat[1][idx2]

print("First 10 values ('bat' in sentence 1):", bat_emb_1[:10])
print("First 10 values ('bat' in sentence 2):", bat_emb_2[:10])

First 10 values ('bat' in sentence 1): tf.Tensor(
[-0.08962515 -0.19197758  0.05445767 -0.23930399  0.06415726 -0.3799652
 -0.03070313 -0.18632221  0.00207844 -0.208526  ], shape=(10,), dtype=float32)
First 10 values ('bat' in sentence 2): tf.Tensor(
[-0.15542574 -0.00922328  0.10498016 -0.23908323 -0.24330795  0.06216738
 -0.05783883  0.32834396 -0.07611501 -0.16495332], shape=(10,), dtype=float32)


**Compare Context (Cosine Similarity)**

In [18]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(bat_emb_1.numpy(), bat_emb_2.numpy())

print("Similarity between 'bat' meanings:", sim)

Similarity between 'bat' meanings: 0.6650806


**TASK2 BERT Model Implementation**

In [19]:
sentences = [
    "The bat is flying",
    "He hit the ball with a bat"
]



In [20]:
# Preprocessing model
preprocess = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

# BERT encoder
bert_model = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3")

**Preprocess Input**

In [21]:

inputs = preprocess(sentences)
print(inputs.keys())



dict_keys(['input_type_ids', 'input_mask', 'input_word_ids'])


**Generate Embeddings**

In [22]:
outputs = bert_model(inputs)

# Word-level embeddings
embeddings = outputs['sequence_output']

print("Embedding shape:", embeddings.shape)

Embedding shape: (2, 128, 768)


**Get Tokens (Important Step)**

In [23]:
# Use tokenizer from preprocess model
tokenizer = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

# Tokenized words (for visualization)
tokens = preprocess(sentences)['input_word_ids']

print(tokens)

tf.Tensor(
[[ 101 1996 7151 2003 3909  102    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0]
 [ 101 2002 2718 1996 3608 2007 1037 7151  102    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    

**Extract bat Embeddings**

In [24]:
bat_emb_1 = embeddings[0][2]  # "bat" in first sentence
bat_emb_2 = embeddings[1][7]  # "bat" in second sentence
print("First 10 values (Sentence 1):", bat_emb_1[:10])
print("First 10 values (Sentence 2):", bat_emb_2[:10])

First 10 values (Sentence 1): tf.Tensor(
[-6.0921535e-05  2.6044559e-01  1.2677921e-01 -2.8091618e-01
  1.3291964e-02 -4.8430112e-01  7.0899099e-01  7.3061723e-01
  2.0883077e-01 -7.8647327e-01], shape=(10,), dtype=float32)
First 10 values (Sentence 2): tf.Tensor(
[-0.06056817 -0.9072482  -0.06744833 -0.27453572 -0.46633813  0.04165878
  0.23526219  0.35728973  0.9171289  -0.6723444 ], shape=(10,), dtype=float32)


**Cosine Similarity**

In [25]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(bat_emb_1.numpy(), bat_emb_2.numpy())

print("Similarity between 'bat' meanings:", sim)

Similarity between 'bat' meanings: 0.43427765


**TASK 2 WITH DIFFERENT TEXT CORPUS**

In [26]:
sentences3=['The bank will not approve the loan',
           'He sat on the river bank']

In [27]:
outputs = bert_model(inputs)

# Word-level embeddings
embeddings = outputs['sequence_output']

print("Embedding shape:", embeddings.shape)

Embedding shape: (2, 128, 768)


**GET TOKENS**

In [28]:
# Use tokenizer from preprocess model
tokenizer = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

# Tokenized words (for visualization)
tokens = preprocess(sentences)['input_word_ids']

print(tokens)

tf.Tensor(
[[ 101 1996 7151 2003 3909  102    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0]
 [ 101 2002 2718 1996 3608 2007 1037 7151  102    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    

**EXTRACT BANK EMBEDDINGS**

In [29]:
bat_emb_1 = embeddings[0][2]  # "bat" in first sentence
bat_emb_2 = embeddings[1][7]  # "bat" in second sentence
print("First 10 values (Sentence 1):", bat_emb_1[:10])
print("First 10 values (Sentence 2):", bat_emb_2[:10])

First 10 values (Sentence 1): tf.Tensor(
[-6.1338767e-05  2.6044559e-01  1.2677912e-01 -2.8091609e-01
  1.3292316e-02 -4.8430163e-01  7.0899099e-01  7.3061734e-01
  2.0883080e-01 -7.8647327e-01], shape=(10,), dtype=float32)
First 10 values (Sentence 2): tf.Tensor(
[-0.06056855 -0.9072489  -0.06744894 -0.274536   -0.46633813  0.04165877
  0.23526274  0.3572897   0.9171283  -0.67234385], shape=(10,), dtype=float32)


**COSINE SIMILARITY**

In [30]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(bat_emb_1.numpy(), bat_emb_2.numpy())

print("Similarity between 'bat' meanings:", sim)

Similarity between 'bat' meanings: 0.43427745


**Text Classification Using ELMO+Naive Bayes**

In [31]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np

from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [32]:
sentences = [
    "Win money now",
    "Claim your prize",
    "Hello how are you",
    "Let's meet tomorrow",
    "Free lottery ticket",
    "Are you coming today"
]


**Load ELMo Model**

In [33]:
elmo = hub.load("https://tfhub.dev/google/elmo/3")


**Generate ELMo Embeddings**

In [34]:
embeddings = elmo.signatures['default'](tf.constant(sentences))['elmo']

print("Shape:", embeddings.shape)

Shape: (6, 4, 1024)


**Convert to Sentence Embeddings**

In [37]:
sentence_embeddings = tf.reduce_mean(embeddings, axis=1)

# Define the labels for the sentences
# Assuming sentences related to offers/prizes are 'spam' (1) and others are 'not spam' (0)
labels = [1, 1, 0, 0, 1, 0]

X = sentence_embeddings.numpy()
y = np.array(labels)

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

Feature shape: (6, 1024)
Label shape: (6,)


**Train-Test Split**

In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



**Train Model**

In [49]:
model = GaussianNB()
model.fit(X_train, y_train)


GaussianNB()

**Model Testing**

In [41]:
y_pred = model.predict(X_test)

print("Predictions:", y_pred)
print("Actual:", y_test)


Predictions: [0 0]
Actual: [1 1]


**Model Evaluation**

In [42]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


Accuracy: 0.0


**Prediction on New Text**

In [43]:
new_sentence = ["Congratulations! You won a free ticket"]

new_emb = elmo.signatures['default'](tf.constant(new_sentence))['elmo']
new_emb = tf.reduce_mean(new_emb, axis=1)

prediction = model.predict(new_emb.numpy())

print("Prediction:", "Spam" if prediction[0] == 1 else "Not Spam")

Prediction: Not Spam


**Text Classification using BERT+NB**

In [44]:
preprocess = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_model = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3")

In [46]:
bert_inputs = preprocess(sentences)

In [47]:
bert_outputs = bert_model(bert_inputs)

# Use sentence embedding ([CLS])
bert_features = bert_outputs['pooled_output'].numpy()

In [48]:
X_train, X_test, y_train, y_test = train_test_split(
    bert_features, labels, test_size=0.2, random_state=42
)

nb_bert = GaussianNB()
nb_bert.fit(X_train, y_train)

y_pred_bert = nb_bert.predict(X_test)
acc_bert = accuracy_score(y_test, y_pred_bert)

print("BERT Accuracy:", acc_bert)


BERT Accuracy: 0.0
